# Data Loading and Raw EEG

## Overview

In this notebook, you will inspect an XDF recording and build an MNE `Raw` object. Most code cells are intentionally incomplete. Read the markdown, replace each `...`, and use the comments as hints. Do not continue until you can explain what the current cell produces.

## Learning objectives

- Explain what an XDF container stores.
- Use metadata and sample properties to identify EEG and marker streams.
- Extract timestamps, sampling rate, labels, and signal values.
- Convert time-by-channel data into an MNE `RawArray`.
- Save an annotated intermediate file.

## Background: What is XDF?

XDF stores multiple time-stamped streams in one file. LabRecorder may capture EEG, PsychoPy/LSL markers, and unrelated streams. Every stream has its own samples, timestamps, and metadata. Stream order is not guaranteed, so selecting stream 0 without inspecting it is unsafe.

In [1]:
# Setup cell: run this as provided.
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import pyxdf

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.eeg_utils import get_stream_channel_names, summarize_xdf_streams

mne.set_log_level("WARNING")

## Step 1: Load the XDF file

1. Confirm that the placeholder path points to your local, de-identified recording.
2. Read the documentation for `pyxdf.load_xdf`.
3. Call it and unpack its two return values into `streams` and `file_header`.
4. Print the number of streams.

**Check:** `streams` should be a list of stream dictionaries.

In [10]:
XDF_PATH = Path("../data/subEvan_ses002_.xdf")

# Check that XDF_PATH exists and give a helpful error.
if not XDF_PATH.exists():
    raise FileNotFoundError(f"Add a de-identified XDF file at {XDF_PATH} or edit XDF_PATH.")

# TODO: load the file and unpack both returned objects.
streams, file_header = pyxdf.load_xdf(str(XDF_PATH))

# TODO: print how many streams were loaded.
print(f"Number of streams loaded: {len(streams)}")

Number of streams loaded: 2


## Step 2: Inspect available streams

An EEG stream usually has many samples, several channels, and a nonzero nominal sampling rate. A marker stream often has one channel, relatively few samples, and a nominal rate of 0 Hz. These are clues, not rules.

Use `summarize_xdf_streams(streams)` to make a table. Then write a loop that prints each stream's index, name, type, channel count, sampling rate, and sample count. Inspect the table before moving on.

In [11]:
# TODO: call the summary helper and display the resulting DataFrame.
stream_summary = summarize_xdf_streams(streams)
display(stream_summary)
# TODO: iterate over rows and print the requested fields.
# Hint: DataFrame.itertuples(index=False) is convenient.
for row in stream_summary.itertuples(index=False):
    print(row.index, row.name, row.type, row.channel_count, row.nominal_srate_hz, row.sample_count)


,index,name,type,channel_count,nominal_srate_hz,sample_count
0,0,WS-default,EEG,24,300.0,282305
1,1,PsychoPyMarkers,Markers,1,0.0,120


0 WS-default EEG 24 300.0 282305
1 PsychoPyMarkers Markers 1 0.0 120


## Step 3: Identify EEG and marker streams

Write down the evidence for your choices before entering the indices. Then select the two dictionaries from `streams`. Print the first 20 marker values and count all markers. If the values look like EEG voltages, you chose the wrong stream.

In [12]:
# STUDENT EDIT: obtain these indices from your stream table.
EEG_STREAM_INDEX = 0
MARKER_STREAM_INDEX = 1

# TODO: select the EEG and marker stream dictionaries.
eeg_stream = streams[0]
marker_stream = streams[1]

# TODO: inspect marker_stream['time_series'].
marker_samples = marker_stream['time_series']
print("Marker count:", len(marker_samples))
print("First 20 markers:", marker_samples[0:20])
pd.Series(marker_samples).value_counts()

Marker count: 120
First 20 markers: [['Right Hand'], ['Left Hand'], ['Left Hand'], ['Right Hand'], ['Left Hand'], ['Left Hand'], ['Right Hand'], ['Right Hand'], ['Right Hand'], ['Right Hand'], ['Left Hand'], ['Left Hand'], ['Left Hand'], ['Right Hand'], ['Left Hand'], ['Right Hand'], ['Left Hand'], ['Right Hand'], ['Right Hand'], ['Left Hand']]


[Right Hand]    60
[Left Hand]     60
Name: count, dtype: int64

## Step 4: Extract EEG data, timestamps, sampling rate, and channel names

XDF EEG samples are normally arranged as samples × channels. Extract `time_series` and `time_stamps`, convert both to NumPy arrays, and read `nominal_srate` from the stream's `info` dictionary. XDF metadata values are often stored inside one-item lists.

Use `get_stream_channel_names`. If its length does not equal the number of data columns, generate neutral names such as `EEG001`. Print the array shape, sampling rate, names, and duration.

In [13]:
# TODO: extract samples and timestamps as floating-point arrays.
eeg_samples = np.array(eeg_stream['time_series'])
eeg_timestamps = np.array(eeg_stream['time_stamps'])
# TODO: extract the nominal sampling rate.
sampling_rate = float(eeg_stream['info']['nominal_srate'][0])
# Optional challenge: if nominal_srate is 0, infer the rate from timestamp differences.
# Hint: 1 / np.median(np.diff(eeg_timestamps))
# TODO: extract exactly one name per data column.
channels = streams[0]['info']['desc'][0]['channels'][0]['channel']
channel_names = [ch['label'][0] for ch in channels]
# TODO: print eeg sample shape, sampling rate, channel names, and recording duration.
recduration = eeg_timestamps[-1] - eeg_timestamps[0]
print(eeg_samples.shape, sampling_rate, channel_names, recduration) 

(282305, 24) 300.0 ['P3', 'C3', 'F3', 'Fz', 'F4', 'C4', 'P4', 'Cz', 'Pz', 'Fp1', 'Fp2', 'T3', 'T5', 'O1', 'O2', 'X3', 'X2', 'F7', 'F8', 'X1', 'A2', 'T6', 'T4', 'TRG'] 941.0228526510764


## Step 5: Create an MNE Raw object

MNE expects channels × time and volts. Determine the XDF unit from metadata or device documentation. Use a scale of `1e-6` only if values are microvolts; use `1.0` if they are already volts.

Create `mne.Info` with `mne.create_info`, transpose and scale the data, then pass both to `mne.io.RawArray`. Finally, convert marker timestamps to seconds relative to the first EEG timestamp and attach them as `mne.Annotations`.

In [23]:
# STUDENT EDIT after checking the recording units.
EEG_SCALE_TO_VOLTS = 1e-6

# TODO: create the MNE Info object.
info = mne.create_info(
    ch_names= channel_names,
    sfreq= sampling_rate,
    ch_types=[ch['type'][0].lower() for ch in channels]
)
# TODO: scale to volts, transpose to channels x time, and create RawArray.
eeg_data_volts = eeg_samples * EEG_SCALE_TO_VOLTS
raw = mne.io.RawArray(eeg_data_volts.T, info, copy='auto')
# TODO: flatten marker values into strings and align onsets to EEG time zero.
# Subtract beginnings from ends of each trial to get duration
durations = 5
markers = [x[0] for x in marker_samples]

new_labels = []
for i in range(0, len(markers)):
    label = markers[i].replace(' ','_').lower()
    new_labels.append(label)

#Subtract initial eeg datastamp from starts of each trial to get onset
marker_onsets = marker_stream['time_stamps'] - eeg_timestamps[0]

annotations = mne.Annotations(
    onset=marker_onsets, duration=durations, description=new_labels
)
raw.set_annotations(annotations)

<mne_qt_browser._pg_figure.MNEQtBrowser(0x22793388900) at 0x00000227AEFF2340>

c:\Users\evana\miniconda3\envs\eeg-analysis-tutorial\Lib\site-packages\mne_qt_browser\_utils.py:36: RuntimeWarning: libpyside: Failed to disconnect (None) from signal "triggered()".
  sig.disconnect()
c:\Users\evana\miniconda3\envs\eeg-analysis-tutorial\Lib\site-packages\mne_qt_browser\_utils.py:36: RuntimeWarning: libpyside: Failed to disconnect (None) from signal "triggered()".
  sig.disconnect()
c:\Users\evana\miniconda3\envs\eeg-analysis-tutorial\Lib\site-packages\mne_qt_browser\_utils.py:36: RuntimeWarning: libpyside: Failed to disconnect (None) from signal "triggered()".
  sig.disconnect()
c:\Users\evana\miniconda3\envs\eeg-analysis-tutorial\Lib\site-packages\mne_qt_browser\_utils.py:36: RuntimeWarning: libpyside: Failed to disconnect (None) from signal "triggered()".
  sig.disconnect()
c:\Users\evana\miniconda3\envs\eeg-analysis-tutorial\Lib\site-packages\mne_qt_browser\_utils.py:36: RuntimeWarning: libpyside: Failed to disconnect (None) from signal "triggered()".
  sig.disconne

## Step 6: Plot the raw EEG

Use `raw.plot` to display about 10 seconds and no more than 20 channels. Start with automatic scaling. Scroll through multiple parts of the recording and verify that annotation labels appear at plausible times. Also calculate each channel's peak-to-peak amplitude in microvolts as a unit sanity check.

In [24]:
# TODO: create the interactive raw-data plot.
raw.plot(
    duration=10,
    n_channels=12,
    scalings='auto',
)

# TODO: compute peak-to-peak amplitude along the time axis and convert V to µV.
peak_to_peak_uv = np.ptp(eeg_samples.T, axis=1)
pd.Series(peak_to_peak_uv, index=raw.ch_names).describe()


count       24.000000
mean     31320.865234
std      15339.690430
min          0.000000
25%      34578.352051
50%      39545.980469
75%      39579.390625
max      39622.734375
dtype: float64

## Step 7: Save the Raw object for later notebooks

Create the output directory if necessary and save with `raw.save`. Use `overwrite=True` only when you intentionally want to replace an earlier result. Confirm the absolute path after saving.

In [25]:
OUTPUT_PATH = Path("../outputs/raw_motor.fif")

# TODO: create the parent directory, save raw, and print the resolved path.
raw.save(OUTPUT_PATH, overwrite=True)

C:\Users\evana\AppData\Local\Temp\ipykernel_2184\251161052.py:4: RuntimeWarning: This filename (c:\Users\evana\Downloads\eeg-analysis-tutorial\Motor_notebooks\..\outputs\raw_motor.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw.save(OUTPUT_PATH, overwrite=True)


[WindowsPath('c:/Users/evana/Downloads/eeg-analysis-tutorial/Motor_notebooks/../outputs/raw_motor.fif')]

## Student exercises

1. Compare the nominal rate with a rate inferred from timestamps.
2. Build a marker-count table with `pandas.Series.value_counts`.
3. Add checks that reject an empty stream or mismatched sample/timestamp counts.

## Reflection questions

- What stream contained EEG, and what evidence supported your choice?
- How many markers were recorded? Was that consistent with 40 trials?
- Were channel names and units present in metadata?
- Why must marker times be aligned to the first EEG sample rather than the first marker?

## Summary

After completing the TODOs, you will have inspected an unknown XDF layout, selected streams using evidence, converted EEG to MNE's expected shape and units, aligned markers, and saved an annotated Raw file.